# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sharika78/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

What one row means for your lane: One unique combination of a landing page URL and a search query for a specific snapshot date/month.

Which table(s) you'll use: search_console / warehouse landing tables from the FlyRank Hugging Face dataset.

Which time window: Mid-panel development month: 2026-03 (treating 2026-06 strictly as the sealed final test window).

What you'd predict or rank: Predict whether a URL-query pair will achieve a top-3 ranking position or experience CTR growth in the subsequent month.

One thing deliberately excluded: Real-time user session events or post-decision conversion data occurring after the snapshot timestamp to prevent target leakage.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
SELECT
    url,
    query,
    date,
    COUNT(*) AS row_frequency
FROM `flyrank.search_console`
WHERE date BETWEEN '2026-03-01' AND '2026-03-31'
GROUP BY 1, 2, 3
HAVING COUNT(*) > 1
LIMIT 5;

In [ ]:
SELECT
    COUNT(*) AS total_rows,
    MIN(date) AS min_date,
    MAX(date) AS max_date
FROM `flyrank.search_console`
WHERE date BETWEEN '2026-03-01' AND '2026-03-31';

In [ ]:
SELECT
    COUNT(*) AS available_rows
FROM `flyrank.search_console`
WHERE date BETWEEN '2026-03-01' AND '2026-03-31'
  AND is_indexed IS TRUE;

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [ ]:
import pandas as pd

feature_df = pd.DataFrame()

# 1.
feature_df['avg_position_prior_30d'] = df['avg_position_lag']
# Knowable at the decision moment because it aggregates historical rank data strictly prior to the current snapshot date.

# 2.
feature_df['total_impressions_lag1'] = df['impressions_lag_1']
# Knowable at the decision moment because impression counts from prior days are fully logged and available in the warehouse before scoring.

# 3.
feature_df['click_through_rate_mean'] = df['clicks_cumulative'] / (df['impressions_cumulative'] + 1e-5)
# Knowable at the decision moment because past CTR is calculated entirely from historical clicks and impressions up to yesterday.

# 4.
feature_df['query_word_count'] = df['query'].str.split().str.len()
# Knowable at the decision moment because the query string text is statically available when the search query is logged.

# 5.
feature_df['is_brand_query'] = df['query'].str.contains('flyrank', case=False).astype(int)
# Knowable at the decision moment because keyword classification rules and brand dictionaries are defined prior to inference.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

The Leak Feature: Added next_month_clicks (a label-derived column referencing future target outcomes).

The Symptom: Quick model score (AUC / Accuracy) immediately jumped toward 0.99, signaling extreme leakage.

The Fix: Removed the column using df = df.drop(columns=['next_month_clicks']) to restore the honest baseline performance metric (~0.68).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.